# 第 7 章 分類モデルの評価指標

正解率が役に立たない場面を出発点に、混同行列・適合率・再現率・F1・AUC を確かめます。

対応する記事: [第 7 章 分類モデルの評価指標（F# 版）](https://github.com/k2works/grokking-machine-learning-excersice/blob/main/docs/article/grokking-machine-learning/fsharp/ch07.md)

実装本体: `apps/grokking-ml-fsharp/src/`

## セットアップ

実装本体（`../src/GrokkingMl/`）を `#load` で読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

VS Code の [Polyglot Notebooks 拡張](https://marketplace.visualstudio.com/items?itemName=ms-dotnettools.dotnet-interactive-vscode) で開くか、Jupyter に .NET Interactive カーネルを登録して実行します。

```bash
dotnet tool install -g Microsoft.dotnet-interactive
dotnet interactive jupyter install
jupyter lab notebooks/
```

In [1]:
#load "../src/GrokkingMl/Ch07Metrics.fs"

open GrokkingMl.Ch07Metrics

## 正解率 99% の役立たずモデル

1000 人に 10 人が罹る病気を、**全員「陰性」と判定する** モデルです。正解率は 99% ですが、病人を 1 人も見つけられません。

**正解率は、陽性と陰性の数が偏っているときに壊れます。**

In [2]:
let sickLabels = List.replicate 10 1 @ List.replicate 990 0
let alwaysHealthy = List.replicate 1000 0

let matrix = confusionMatrix sickLabels alwaysHealthy
printfn "%A" matrix
printfn "正解率 %.3f" (accuracy matrix)
printfn "再現率 %.3f  ← 病人を 1 人も見つけられていない" (recall matrix)

{ TruePositives = 0
  FalsePositives = 0
  FalseNegatives = 10
  TrueNegatives = 990 }

正解率 

0.990

再現率 

0.000

  ← 病人を 1 人も見つけられていない

## 適合率と再現率はトレードオフ

逆に **全員を「陽性」と判定** すると、見逃しはゼロ（再現率 1.0）ですが、陽性と言った 1000 件のうち当たりは 10 件だけです。**片方の指標だけを追うと、必ずもう一方が壊れます。**

In [3]:
let aggressive = confusionMatrix sickLabels (List.replicate 1000 1)

printfn "%-12s %8s %8s %8s %8s" "モデル" "正解率" "適合率" "再現率" "F1"

for (name, m) in [ "全員陰性", matrix; "全員陽性", aggressive ] do
    printfn "%-12s %8.3f %8.3f %8.3f %8.3f" name (accuracy m) (precision m) (recall m) (f1Score m)

モデル         

     正解率

     適合率

     再現率

      F1

...

全員陽性        

   0.010

   0.010

   1.000

   0.020

## F1 は調和平均

適合率 1.0・再現率 0.1 のモデルは、算術平均なら 0.55 と「まあまあ」に見えます。**F1 は 0.18 です。片方が壊れているモデルを、平均で誤魔化させません。**

In [4]:
let unbalanced =
    { TruePositives = 1; FalsePositives = 0; FalseNegatives = 9; TrueNegatives = 90 }

printfn "適合率 %.3f  再現率 %.3f" (precision unbalanced) (recall unbalanced)
printfn "算術平均 %.3f" ((precision unbalanced + recall unbalanced) / 2.0)
printfn "F1       %.3f  ← 偏りを強く罰する" (f1Score unbalanced)

適合率 

1.000

  再現率 

0.100

算術平均 

0.550

F1       

0.182

  ← 偏りを強く罰する

## F ベータで重視する側を選ぶ

病気の見逃しを避けたいなら `beta > 1`（再現率重視）、迷惑メール判定で誤検知を避けたいなら `beta < 1`（適合率重視）です。**指標そのものを目的に合わせて調整できます。**

In [5]:
let sample =
    { TruePositives = 3; FalsePositives = 1; FalseNegatives = 2; TrueNegatives = 4 }

printfn "適合率 %.3f  再現率 %.3f" (precision sample) (recall sample)

for beta in [ 0.5; 1.0; 2.0 ] do
    printfn "F%.1f = %.4f" beta (fBetaScore beta sample)

適合率 

0.750

  再現率 

0.600

F

...

0.6667

F

2.0

 = 

0.6250

## AUC は閾値に依存しない

AUC は **「陽性を陰性より高くランク付けできた組の割合」** です。確率が両極に分かれていても中央に固まっていても、**順位が同じなら AUC は同じ** になります。

In [6]:
let cases =
    [ "完全な順位", [ 1; 1; 0; 0 ], [ 0.9; 0.8; 0.2; 0.1 ]
      "3/4 が正しい順", [ 1; 0; 1; 0 ], [ 0.8; 0.6; 0.4; 0.2 ]
      "情報なし", [ 1; 0; 0; 1 ], [ 0.8; 0.6; 0.4; 0.2 ]
      "完全に逆", [ 0; 0; 1; 1 ], [ 0.9; 0.8; 0.2; 0.1 ] ]

for (name, ls, ps) in cases do
    printfn "%-16s AUC = %.3f" name (auc ls ps)

printfn ""
printfn "確率を圧縮しても AUC は変わらない:"
printfn " 両極 %f" (auc [ 1; 1; 0; 0 ] [ 0.99; 0.98; 0.02; 0.01 ])
printfn " 中央 %f" (auc [ 1; 1; 0; 0 ] [ 0.55; 0.54; 0.46; 0.45 ])

完全な順位           

 AUC = 

1.000

3/4 が正しい順       

 AUC = 

...

確率を圧縮しても AUC は変わらない:

 両極 

1.000000

 中央 

1.000000

## 試してみる: 閾値を動かす

**閾値はモデルの性能ではなく、運用上の選択です。** 下げれば再現率が上がり、適合率が下がります。

In [7]:
let labels = [ 1; 1; 0; 0 ]
let probabilities = [ 0.9; 0.4; 0.6; 0.1 ]

printfn "%6s %-16s %8s %8s" "閾値" "予測" "適合率" "再現率"

for threshold in [ 0.2; 0.5; 0.8 ] do
    let predictions = predictionsAtThreshold threshold probabilities
    let m = confusionMatrix labels predictions
    printfn "%6.1f %-16s %8.3f %8.3f" threshold (sprintf "%A" predictions) (precision m) (recall m)

    閾値

予測              

     適合率

     再現率

   0.2

...

   0.500

   0.8

[1; 0; 0; 0]    

   1.000

   0.500